## Library Imports

This notebook uses standard Python libraries for data preprocessing, regression modeling, cross-validation, and performance evaluation.

- **Data Handling:** NumPy, Pandas  
- **Preprocessing:** StandardScaler  
- **Models:** Linear Regression, Random Forest, Gradient Boosting, XGBoost  
- **Validation & Tuning:** K-Fold Cross Validation, RandomizedSearchCV  
- **Evaluation Metrics:** RMSE, MAE, R²  

Warnings are suppressed to keep the notebook output clean.


In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor

from sklearn.model_selection import KFold,cross_val_score,RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

## Loading Preprocessed Dataset

The cleaned and feature-engineered dataset is loaded to ensure consistency across modeling experiments.
This version contains handled missing values, treated outliers, transformed features, and is ready for machine learning workflows.


In [2]:
df=pd.read_csv("/content/solar_preprocessed_data.csv")

## Outlier Treatment

Outliers in key numerical features are handled using the **Interquartile Range (IQR) method**.

For each selected column:
- Q1 (25th percentile) and Q3 (75th percentile) are computed  
- IQR is calculated as `Q3 − Q1`  
- Values outside the range `Q1 − 1.5 × IQR` and `Q3 + 1.5 × IQR` are **capped** to the respective bounds  

This approach reduces the impact of extreme values while preserving the overall data distribution.


In [3]:
outlier_cols = [
    'Estimated PV System Size (kWdc)',
    'PV System Size (kWac)',
    'Estimated Annual PV Energy Production (kWh)'
]

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower_bound, lower_bound,
                       np.where(df[col] > upper_bound, upper_bound, df[col]))


## Outlier Analysis (Statistical Summary)

This step examines the statistical distribution of numerical features identified as potential outlier candidates.  
Descriptive statistics are reviewed to understand range, spread, and extreme values before applying any outlier treatment.


In [4]:
for col in outlier_cols:
    print(col)
    print(df[col].describe())
    print("-"*40)


Estimated PV System Size (kWdc)
count    218115.000000
mean          8.203566
std           3.941929
min           0.010000
25%           5.270000
50%           7.130000
75%          10.240000
max          17.695000
Name: Estimated PV System Size (kWdc), dtype: float64
----------------------------------------
PV System Size (kWac)
count    218115.000000
mean          7.011595
std           3.369395
min           0.010000
25%           4.500000
50%           6.090000
75%           8.750000
max          15.125000
Name: PV System Size (kWac), dtype: float64
----------------------------------------
Estimated Annual PV Energy Production (kWh)
count    218115.000000
mean       9629.577821
std        4627.516039
min          14.000000
25%        6180.000000
50%        8364.000000
75%       12017.000000
max       20772.500000
Name: Estimated Annual PV Energy Production (kWh), dtype: float64
----------------------------------------


## Skewed Feature Identification

The following numerical features exhibit skewed distributions and require special handling during preprocessing (e.g., scaling or transformation):

- Estimated PV System Size (kWdc)  
- PV System Size (kWac)  
- Estimated Annual PV Energy Production (kWh)

Identifying skewed features helps improve model stability and predictive performance.


In [5]:
skewed_cols = [
    'Estimated PV System Size (kWdc)',
    'PV System Size (kWac)',
    'Estimated Annual PV Energy Production (kWh)'
]


## Skewness Treatment

Log transformation (`log1p`) is applied to skewed numerical features to reduce skewness and stabilize variance.  
This helps improve model performance by making the data distribution closer to normal.

Post-transformation skewness is evaluated to verify the effectiveness of the treatment.


In [6]:
for col in skewed_cols:
    df[col] = np.log1p(df[col])
df[skewed_cols].skew()


,0
Estimated PV System Size (kWdc),-0.070563
PV System Size (kWac),-0.043799
Estimated Annual PV Energy Production (kWh),-0.372730


## Interconnection Date Processing

The *Interconnection Date* column is converted to datetime format to ensure consistency.  
The year of interconnection is extracted as a new feature for analysis, and the original date column is removed as it is no longer required.


In [7]:
df['Interconnection Date'] = pd.to_datetime(df['Interconnection Date'], errors='coerce')
df['Interconnection_Year'] = df['Interconnection Date'].dt.year

df.drop(columns=['Interconnection Date'], inplace=True)


## Feature Engineering: Storage Indicator

A binary feature **Has_Storage** is created to indicate whether a system includes an energy storage component.

- Value **1** → Energy storage system present  
- Value **0** → No energy storage system  

The original *Energy Storage System Size (kWac)* column is dropped after feature creation to avoid redundancy and reduce dimensionality.


In [8]:
df['Has_Storage'] = np.where(df['Energy Storage System Size (kWac)'] > 0, 1, 0)

df.drop(columns=['Energy Storage System Size (kWac)'], inplace=True)


## Rare Category Grouping

Low-frequency categories in high-cardinality categorical features are grouped into a single **"Other"** category.  
This helps reduce noise, improve model stability, and prevent overfitting.

Applied to:
- **Developer**
- **City/Town**


In [9]:
def group_rare(series, min_freq=0.01):
    freq = series.value_counts(normalize=True)
    rare = freq[freq < min_freq].index
    return series.replace(rare, 'Other')

df['Developer'] = group_rare(df['Developer'])
df['City/Town'] = group_rare(df['City/Town'])


## Reviewing Engineered Dataset Structure

This step verifies the final dataset after feature engineering by:

- Checking data types and non-null counts for all features  
- Confirming successful removal of non-informative columns  
- Ensuring newly created features are correctly added  
- Previewing sample records for validation  

This quality check ensures the dataset is fully prepared for machine learning modeling.


In [10]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218115 entries, 0 to 218114
Data columns (total 11 columns):
 #   Column                                       Non-Null Count   Dtype  
---  ------                                       --------------   -----  
 0   Utility                                      218115 non-null  object 
 1   City/Town                                    218115 non-null  object 
 2   County                                       218115 non-null  object 
 3   Zip                                          218115 non-null  float64
 4   Developer                                    218115 non-null  object 
 5   Metering Method                              218115 non-null  object 
 6   Estimated PV System Size (kWdc)              218115 non-null  float64
 7   PV System Size (kWac)                        218115 non-null  float64
 8   Estimated Annual PV Energy Production (kWh)  218115 non-null  float64
 9   Interconnection_Year                         218115 non-nul

,Utility,City/Town,County,Zip,Developer,Metering Method,Estimated PV System Size (kWdc),PV System Size (kWac),Estimated Annual PV Energy Production (kWh),Interconnection_Year,Has_Storage
0,Con Ed,Other,Queens,11418.0,Kamtech Solar Solutions,NM,1.953028,1.819699,8.867991,2023,1
1,Con Ed,Bronx,Bronx,10473.0,Kamtech Solar Solutions,NM,2.046402,1.911023,8.976136,2023,1
2,Con Ed,Brooklyn,Kings,11225.0,SUNCO,NM,1.398717,1.283708,8.184793,2023,1
3,Con Ed,Brooklyn,Kings,11236.0,Kamtech Solar Solutions,NM,1.890095,1.757858,8.793764,2023,1
4,Con Ed,Other,Queens,11413.0,Kamtech Solar Solutions,NM,1.953028,1.819699,8.867991,2023,1


## Verifying Missing Values After Feature Engineering

New features created during transformation may introduce missing or infinite values.  
This step rechecks the dataset to identify any remaining missing values across all columns and ensures the dataset is fully clean before model training.


In [11]:
df.isnull().sum().sort_values(ascending=False)


,0
Utility,0
City/Town,0
County,0
Zip,0
Developer,0
Metering Method,0
Estimated PV System Size (kWdc),0
PV System Size (kWac),0
Estimated Annual PV Energy Production (kWh),0
Interconnection_Year,0


## Feature and Target Separation

The dataset is divided into input features (`X`) and the target variable (`y`) for regression modeling.

- **Target Variable:** Estimated Annual PV Energy Production (kWh)  
- **Features:** All remaining columns used as predictors

This separation prepares the data for model training and evaluation.


In [12]:

target = 'Estimated Annual PV Energy Production (kWh)'

X = df.drop(columns=[target])
y = df[target]


## Categorical Feature Encoding

Categorical variables are converted into numerical format using **one-hot encoding** to make them compatible with machine learning models.

- `get_dummies()` is used to create binary columns for each category  
- `drop_first=True` is applied to avoid multicollinearity (dummy variable trap)


In [13]:
X = pd.get_dummies(X, drop_first=True)


## Train–Test Split

The dataset is split into training and testing sets to evaluate model performance on unseen data.

- **Training Set:** 80%  
- **Testing Set:** 20%  
- **Random State:** 42 (for reproducibility)


In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


## Feature Scaling

Numerical features are standardized using **StandardScaler** to ensure all variables are on a common scale.  
The scaler is **fit on the training data only** and then applied to the test data to avoid data leakage and ensure fair model evaluation.


In [15]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## Model Evaluation Function

This function calculates key regression performance metrics used to assess model accuracy and generalization:

### Metrics Used:
- **RMSE (Root Mean Squared Error)** – Penalizes large prediction errors and reflects overall model accuracy  
- **MAE (Mean Absolute Error)** – Measures average prediction deviation in original units  
- **R² Score** – Indicates the proportion of variance in energy production explained by the model  

These metrics together provide a comprehensive evaluation of predictive performance.


In [16]:
def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2


## Cross-Validation Strategy

K-Fold Cross Validation is used to evaluate model performance in a robust and unbiased manner.

- **Number of folds:** 3 (balances speed and reliability)
- **Shuffling:** Enabled to ensure random data distribution
- **Random State:** Fixed for reproducibility


In [17]:
kf = KFold(
    n_splits=3,          # fast & sufficient
    shuffle=True,
    random_state=42
)


## Linear Regression – Cross-Validation Evaluation

Linear Regression is used as a baseline model to evaluate the relationship between features and the target variable.

- **Validation Technique:** K-Fold Cross Validation  
- **Metric Used:** Root Mean Squared Error (RMSE)  
- **Purpose:** Establish a benchmark performance for comparison with advanced models


In [20]:
lr = LinearRegression()

lr_cv_rmse = -cross_val_score(
    lr,
    X_train_scaled,
    y_train,
    scoring='neg_root_mean_squared_error',
    cv=kf,
    n_jobs=-1
).mean()


linear model cv_rmse value 0.013190889229880974


## Linear Regression – Model Evaluation

The Linear Regression model is trained on scaled features and evaluated on the test dataset.  
Performance metrics are reported using cross-validation and test set predictions.

**Reported Metrics:**
- Cross-Validation RMSE
- Test RMSE
- Mean Absolute Error (MAE)
- R² Score


In [21]:
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)

lr_rmse, lr_mae, lr_r2 = evaluate_model(y_test, lr_pred)

print("linear regression model results")
print("CV_RMSE :", lr_cv_rmse)
print("RMSE value :", lr_rmse)
print("MAE value :", lr_mae)
print("R2 value :", lr_r2)


linear regression model results
CV_RMSE : 0.013190889229880974
RMSE value : 0.022743521917752606
MAE value : 0.005869103626805322
R2 value : 0.9979056748893806


## Random Forest Model – Baseline Evaluation

A **Random Forest Regressor** is used as a baseline ensemble model to capture non-linear relationships and feature interactions.

- The model is configured with fixed hyperparameters for stability and reproducibility.
- **No feature scaling is applied**, as Random Forest is not sensitive to feature magnitude.
- **K-Fold cross-validation** is used to evaluate model performance and reduce overfitting.
- Model performance is measured using **Root Mean Squared Error (RMSE)**.

This step establishes a reliable benchmark for comparison with other regression models.


In [22]:
rf_base = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

rf_cv_rmse = -cross_val_score(
    rf_base,
    X_train,        # ⚠ RF does NOT need scaling
    y_train,
    scoring='neg_root_mean_squared_error',
    cv=kf,
    n_jobs=-1
).mean()


## Random Forest – Model Evaluation

The Random Forest model is trained on the training dataset and evaluated on the test set to assess its predictive performance.

Model performance is reported using:
- **Cross-Validated RMSE** for baseline stability
- **Test RMSE** to measure prediction error
- **Test MAE** for average absolute deviation
- **Test R²** to evaluate explained variance


In [29]:
rf_base.fit(X_train, y_train)

rf_pred = rf_base.predict(X_test)
rf_rmse, rf_mae, rf_r2 = evaluate_model(y_test, rf_pred)

print("random forest model performance")
print(f"Baseline RF CV RMSE : {rf_cv_rmse:.6f}")
print(f"Test RMSE           : {rf_rmse:.6f}")
print(f"Test MAE            : {rf_mae:.6f}")
print(f"Test R2             : {rf_r2:.6f}")


random forest model performance
Baseline RF CV RMSE : 0.003781
Test RMSE           : 0.005911
Test MAE            : 0.000051
Test R2             : 0.999859


## Gradient Boosting – Hyperparameter Tuning

RandomizedSearchCV is used to tune the Gradient Boosting Regressor by optimizing key parameters such as the number of estimators, learning rate, tree depth, and subsampling rate.

The model is evaluated using K-Fold cross-validation with **RMSE** as the scoring metric to ensure robust performance.


In [30]:
gb = GradientBoostingRegressor(random_state=42)

gb_params = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [2, 3],
    'subsample': [0.8, 1.0]
}

gb_search = RandomizedSearchCV(
    gb,
    gb_params,
    n_iter=8,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)




## Gradient Boosting Model – Training & Evaluation

Hyperparameter tuning is performed using cross-validation to identify the optimal Gradient Boosting configuration.  
The selected model is then evaluated on the test dataset using standard regression metrics to assess generalization performance.


In [33]:
gb_search.fit(X_train, y_train)

best_gb = gb_search.best_estimator_
gb_cv_rmse = -gb_search.best_score_

gb_pred = best_gb.predict(X_test)
gb_rmse, gb_mae, gb_r2 = evaluate_model(y_test, gb_pred)
print("Gradient Boosting model performance")
print(f"Baseline RF CV RMSE : {gb_cv_rmse:.6f}")
print(f"Test RMSE           : {gb_rmse:.6f}")
print(f"Test MAE            : {gb_mae:.6f}")
print(f"Test R2             : {gb_r2:.6f}")


random forest model performance
Baseline RF CV RMSE : 0.005151
Test RMSE           : 0.002785
Test MAE            : 0.001579
Test R2             : 0.999969


## XGBoost Hyperparameter Tuning

XGBoost Regressor is optimized using **RandomizedSearchCV** to identify the best-performing hyperparameters while keeping computation efficient.

- **Objective:** Regression with squared error loss  
- **Cross-Validation:** K-Fold strategy  
- **Evaluation Metric:** RMSE (negative RMSE used for optimization)  
- **Search Strategy:** Randomized search to balance performance and training time  

This approach ensures improved generalization and avoids overfitting.


In [34]:
xgb = XGBRegressor(
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

xgb_params = {
    'n_estimators': [300, 500],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_search = RandomizedSearchCV(
    xgb,
    xgb_params,
    n_iter=8,
    cv=kf,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1
)



## XGBoost Model Training & Evaluation

The XGBoost regression model is trained using hyperparameters optimized through **RandomizedSearchCV**.  
Cross-validation performance is evaluated using **RMSE**, and the best-performing estimator is selected.

The final model is then tested on unseen data to assess:
- **Test RMSE** – overall prediction error
- **Test MAE** – average absolute error
- **Test R² Score** – goodness of fit

This step ensures the model generalizes well and avoids overfitting.


In [36]:
xgb_search.fit(X_train_scaled, y_train)

best_xgb = xgb_search.best_estimator_
xgb_cv_rmse = -xgb_search.best_score_

xgb_pred = best_xgb.predict(X_test_scaled)
xgb_rmse, xgb_mae, xgb_r2 = evaluate_model(y_test, xgb_pred)

print("XGBoost model performance")
print(f"Baseline RF CV RMSE : {xgb_cv_rmse:.6f}")
print(f"Test RMSE           : {xgb_rmse:.6f}")
print(f"Test MAE            : {xgb_mae:.6f}")
print(f"Test R2             : {xgb_r2:.6f}")


XGBoost model performance
Baseline RF CV RMSE : 0.032156
Test RMSE           : 0.040834
Test MAE            : 0.004288
Test R2             : 0.993249


## Gradient Boosting – Hyperparameter Tuning Results

The optimal hyperparameters for the Gradient Boosting Regressor were identified using **RandomizedSearchCV** with cross-validation.  
The model performance is evaluated using **RMSE**, where a lower value indicates better predictive accuracy.

- **Best Parameters:** Displayed below  
- **Best Cross-Validated RMSE:** Reported from the tuning process  

These parameters are used to train the final Gradient Boosting model.


In [39]:
print("Best Gradient Boosting Parameters:")
for param, value in gb_search.best_params_.items():
    print(f"{param}: {value}")

print("\nBest CV RMSE:", -gb_search.best_score_)


Best Gradient Boosting Parameters:
subsample: 0.8
n_estimators: 200
max_depth: 3
learning_rate: 0.1

Best CV RMSE: 0.005151121575027843


## Model Performance Comparison

The table below summarizes the performance of all regression models evaluated using cross-validation and test data.

Models are compared based on:
- **RMSE** – Overall prediction error (lower is better)
- **MAE** – Average absolute error
- **R² Score** – Variance explained by the model

The results are sorted by **RMSE** to identify the best-performing model.


In [37]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost'],
    'CV_RMSE': [lr_cv_rmse, rf_cv_rmse, gb_cv_rmse, xgb_cv_rmse],
    'RMSE': [lr_rmse, rf_rmse, gb_rmse, xgb_rmse],
    'MAE': [lr_mae, rf_mae, gb_mae, xgb_mae],
    'R2 Score': [lr_r2, rf_r2, gb_r2, xgb_r2]
})

results.sort_values(by='RMSE')


,Model,CV_RMSE,RMSE,MAE,R2 Score
2,Gradient Boosting,0.005151,0.002785,0.001579,0.999969
1,Random Forest,0.003781,0.005911,0.000051,0.999859
0,Linear Regression,0.013191,0.022744,0.005869,0.997906
3,XGBoost,0.032156,0.040834,0.004288,0.993249
